In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from ripser import ripser

from persim import plot_diagrams
from persim import PersLandscapeApprox
from gudhi.representations import BettiCurve

def standard_plot_barcodes(dgms, ax, title):
    y_base = 0
    y_ticks = []
    y_tick_labels = []

    colors = plt.colormaps['tab10'].colors

    for dim, dgm in enumerate(dgms):
        if len(dgm) == 0:
            continue

        dgm_sorted = dgm[np.argsort(dgm[:, 0])]
        finite_deaths = dgm_sorted[dgm_sorted[:, 1] != np.inf][:, 1] if len(dgm_sorted) > 0 else []
        max_death = np.max(finite_deaths) if len(finite_deaths) > 0 else 2.0
        inf_length = max(max_death * 1.2, 1.0)

        for birth, death in dgm_sorted:
            if np.isinf(death) or death > 1e10:
                ax.plot([birth, inf_length], [y_base, y_base], color=colors[dim % len(colors)], lw=2, linestyle='--')
            else:
                ax.plot([birth, death], [y_base, y_base], color=colors[dim % len(colors)], lw=2)
            y_base += 1

        y_ticks.append(y_base - len(dgm)/2)
        y_tick_labels.append(f"H{dim}")
        y_base += 2

    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_tick_labels)
    ax.set_title(title)
    ax.set_xlabel("Filtration (Epsilon)")

def plot_landscape_simple(pla, ax, alpha=0.8, title="Landscape", depth_range=range(4)):
    t_vals = np.linspace(pla.start, pla.stop, pla.num_steps)
    for depth in depth_range:
        if depth < len(pla.values):
            ax.plot(t_vals, pla.values[depth], label=f"Layer {depth+1}", alpha=alpha, lw=2)
            
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel("Filtration (Epsilon)")
    ax.set_ylabel("Landscape Value")
    ax.legend()


output_dir = os.path.join('plots', 'visuals')
os.makedirs(output_dir, exist_ok=True)

np.random.seed(42)
n_inner = 150
inner_points = np.random.randn(n_inner, 2) * 0.7
n_outer = 250
theta = np.random.uniform(0, 2 * np.pi, n_outer)
r = np.random.uniform(4, 5.5, n_outer)
outer_points = np.column_stack((r * np.cos(theta), r * np.sin(theta)))

data = np.vstack((inner_points, outer_points))

res = ripser(data, maxdim=1)
dgms = res['dgms']


plt.figure(figsize=(8, 8), dpi=150)
plt.scatter(data[:, 0], data[:, 1], s=20, c='black')
plt.title('Raw Data', fontsize=16, fontweight='bold', pad=15)
plt.grid(True, linestyle='--', alpha=0.5)
plt.gca().set_aspect('equal')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'plot_1_raw_data.png'), bbox_inches='tight')
plt.close()


fig, ax = plt.subplots(figsize=(8, 8), dpi=150)
safe_dgms = []
for d in range(len(dgms)):
    safe_dgm = np.copy(dgms[d])
    if len(safe_dgm) == 0:
        safe_dgm = np.array([[0.0, 0.0001], [0.0, np.inf]])
    safe_dgms.append(safe_dgm)
        
plot_diagrams(safe_dgms, show=False, ax=ax)
ax.set_title("Persistence Diagram", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'plot_2_persistence_diagram.png'), bbox_inches='tight')
plt.close()


fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
standard_plot_barcodes(dgms, ax, "Persistence Barcodes")
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'plot_3_persistence_barcode.png'), bbox_inches='tight')
plt.close()


fig_bc, ax_bc = plt.subplots(figsize=(10, 6), dpi=150)


plot_colors = {0: '#1f77b4', 1: '#d62728'}

for hom_deg in [0, 1]:
    if hom_deg < len(dgms):
        dgm_for_deg = dgms[hom_deg]
        finite_dgm = dgm_for_deg[dgm_for_deg[:, 1] != np.inf]
        
        if len(finite_dgm) > 0:
            min_b, max_d = np.min(finite_dgm[:, 0]), np.max(finite_dgm[:, 1])
            
            bc_transformer = BettiCurve(resolution=500, sample_range=[min_b, max_d])
            betti_vals = bc_transformer.fit_transform([finite_dgm])[0]
            t_vals = np.linspace(min_b, max_d, 500)
            
            ax_bc.plot(t_vals, betti_vals, linewidth=2.5, color=plot_colors[hom_deg], label=f'$H_{hom_deg}$')
            ax_bc.fill_between(t_vals, betti_vals, alpha=0.15, color=plot_colors[hom_deg])

ax_bc.set_title('Betti Curves ($H_0$ & $H_1$)', fontsize=16, fontweight='bold')
ax_bc.set_xlabel(r"Filtration Parameter ($\epsilon$)", fontsize=12)
ax_bc.set_ylabel("Betti Number", fontsize=12)
ax_bc.grid(True, linestyle='--')
ax_bc.legend(fontsize=12)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'plot_4_betti_curves.png'), bbox_inches='tight', transparent=True)
plt.close()


for hom_deg in [1, 2]:
    fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
    
    if hom_deg < len(dgms):
        dgm_for_deg = dgms[hom_deg]
        finite_dgm = dgm_for_deg[dgm_for_deg[:, 1] != np.inf]
        
        if len(finite_dgm) > 0:
            pla = PersLandscapeApprox(dgms=[finite_dgm], hom_deg=0)
            
            plot_landscape_simple(
                pla, 
                ax=ax, 
                alpha=0.7,
                title=f'$H_{hom_deg}$ Landscape', 
                depth_range=range(4)
            )
            ax.grid(True, linestyle='--', alpha=0.5)
        else:
            ax.set_title(f'$H_{hom_deg}$ - No finite features')
            ax.axis('off')
    else:
        ax.set_title(f'$H_{hom_deg}$ - Dimension not computed (check maxdim in ripser)')
        ax.axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'plot_5_persistent_landscape_H{hom_deg}.png'), bbox_inches='tight')
    plt.close()